# Visualizacioes para Seguimiento a embarques
- V1. 2025-02-08
    - Version inicial

In [1]:
import pandas as pd
import panel as pn
import plotly.express as px
import os
import pickle

pn.extension("plotly")

def load_state_pickle(filename='folder_state_local_shipments.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}
    
state = load_state_pickle()

path_graph_data = os.path.join(state['folder_output'], 'graph_data.xlsx')

if not os.path.exists(path_graph_data):
    print(f'No se encontró el archivo de datos: {path_graph_data}')
    raise SystemExit()

df = pd.read_excel(path_graph_data)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Widgets
grouping_select = pn.widgets.Select(name="Group By", options=["Raw", "Month"], value="Month")
date_range = pn.widgets.DateRangeSlider(
    name="Date Range",
    start=df['Date'].min(),
    end=df['Date'].max(),
    value=(df['Date'].min(), df['Date'].max())
)
refresh_button = pn.widgets.Button(name="Refresh", button_type="primary")

def get_pivot_data(grouping, date_range):
    start_date = pd.Timestamp(date_range[0])
    end_date = pd.Timestamp(date_range[1])
    filtered_df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
    
    if grouping == "Month":
        df_grouped = filtered_df.copy()
        df_grouped["Group"] = df_grouped["Date"].dt.to_period("M").astype(str)
        pivot = df_grouped.pivot_table(index="Group", columns="Category", values="Cum Sum", aggfunc="max")
    else:
        pivot = filtered_df.pivot_table(index="Date", columns="Category", values="Cum Sum", aggfunc="max")
    
    return pivot

def update_bar_chart(grouping, date_range):
    pivot = get_pivot_data(grouping, date_range)
    df_pivot = pivot.reset_index()
    melt_col = df_pivot.columns[0]
    df_melt = df_pivot.melt(id_vars=melt_col, var_name="Category", value_name="Cum Sum")
    
    df_melt['Category'] = pd.Categorical(
        df_melt['Category'], 
        categories=['EDI', 'Shipped CUU', 'Shipped Elp'], 
        ordered=True
    )
    
    fig = px.bar(
        df_melt,
        x=melt_col,
        y="Cum Sum",
        color="Category",
        title="Cum Sum by Category",
        category_orders={"Category": ['EDI', 'Shipped ELP', 'Shipped Cust']}
    )

    fig.update_layout(barmode="group")
    
    return pn.pane.Plotly(fig, config={"displayModeBar": True}, sizing_mode='scale_height')

def update_pie_chart(date_range):
    start_date = pd.Timestamp(date_range[0])
    end_date = pd.Timestamp(date_range[1])
    filtered_df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
    
    pie_data = filtered_df.groupby("Category")["Quantity"].sum().reset_index()
    fig_pie = px.pie(pie_data, names="Category", values="Quantity", title="Total Quantity by Category")
    return pn.pane.Plotly(fig_pie, config={"displayModeBar": True}, sizing_mode='scale_height')

def update_table(grouping, date_range):
    pivot = get_pivot_data(grouping, date_range)
    return pn.pane.DataFrame(pivot.T.reset_index())

# Bind widgets to update functions
bar_chart = pn.bind(update_bar_chart, grouping=grouping_select, date_range=date_range)
pie_chart = pn.bind(update_pie_chart, date_range=date_range)
table_view = pn.bind(update_table, grouping=grouping_select, date_range=date_range)

def refresh(event):
    grouping_select.param.trigger('value')
    date_range.param.trigger('value')

refresh_button.on_click(refresh)

# Adjusted Layout for Full Visibility on Screen

controls_row = pn.Row(
    pn.Column(grouping_select, date_range, refresh_button, sizing_mode="stretch_width"),
    sizing_mode="stretch_width"
)

charts_row = pn.Row(
    pn.Column(grouping_select, date_range, refresh_button,
        bar_chart,
         styles={"min-height": "65vh"}
    ),
    pn.Column(
        pie_chart,
         styles={"min-height": "65vh"}
    ),
    sizing_mode="stretch_width"
)

table_row = pn.Row(
    table_view,
    sizing_mode="stretch_width",
    styles={"min-height": "20vh", "overflow": "auto"}  # Ensures the table fits within screen
)

template = pn.template.FastListTemplate(
    title="Shipping follow up",
    header_background="#4b566b",
    main=[
        # controls_row,
        charts_row,
        table_row
    ]
)

template.servable()


FastListTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [main-2447044610672] Row(sizing_mode='stretch_width')
        [0] Column(styles={'min-height': '65vh'})
            [0] Select(name='Group By', options=['Raw', 'Month'], value='Month')
            [1] DateRangeSlider(end=Timestamp('2025-02-08 0..., name='Date Range', start=Timestamp('2024-01-10 0..., value=(Timestamp('2024-01-10 00:..., value_end=Timestamp('2025-02-08 0..., value_start=Timestamp('2024-01-10 0...)
            [2] Button(button_type='primary', name='Refresh')
            [3] ParamFunction(function, _pane=Plotly, defer_load=False)
        [1] Column(styles={'min-height': '65vh'})
            [0] ParamFunction(function, _pane=Plotly, defer_load=False)
    [main-2447070362608] Row(sizing_mode='stretch_width', styles={'min-height': '20vh', ...})
        [0] ParamFunction(function, _pane=DataFrame, defer_load=False)